In [41]:
import freetype
from PIL import Image
import os

face = freetype.Face("/Applications/HEXPLODATIONS/Utopia-Regular_8_dfont/Utopia-Regular_8_1984_000007C0.dfont")

# Какие битмап-страйки есть в шрифте
print("Available strikes:")
for i, s in enumerate(face.available_sizes):
    print(f"  [{i}] {s.width}×{s.height}px  (size={s.size/64:.1f}pt, ppem={s.x_ppem/64:.1f})")

# Берём первый страйк
face.select_size(0)

def render_glyph(char, face):
    # Рендерим букву A в mono
    face.load_char(char, freetype.FT_LOAD_RENDER | freetype.FT_LOAD_TARGET_MONO)
    bmp = face.glyph.bitmap
    w, h, pitch = bmp.width, bmp.rows, abs(bmp.pitch)

    print(f"\nglyph '{char}': {w}×{h}px, pitch={pitch}, "
        f"buffer={len(bmp.buffer)} байт, pixel_mode={bmp.pixel_mode}")

    # Распаковываем 1-бит-на-пиксель -> 8-бит-на-пиксель (0 / 255)
    data = bytearray([255] * (w * h))
    for y in range(h):
        for x in range(w):
            byte = bmp.buffer[y * pitch + (x >> 3)]
            if byte & (0x80 >> (x & 7)):
                data[y * w + x] = 0

    img = Image.frombytes("L", (w, h), bytes(data))
    return img

Available strikes:
  [0] 0×8px  (size=8.0pt, ppem=8.0)


In [42]:
def make_grid(face):
    chars = ['A', 'B', 'C', 'a', 'b', 'c', '0', '1', '2']
    imgs = []
    for char in chars:
        img = render_glyph(char, face)
        imgs.append(img)

    max_w = max(img.width for img in imgs)
    max_h = max(img.height for img in imgs)

    gap = 2
    cols = 3
    rows = (len(imgs) + cols - 1) // cols

    cell_w = max_w + gap
    cell_h = max_h + gap

    # Create a white background grid
    grid = Image.new("L", (cols * cell_w, rows * cell_h), color=255)

    for i, img in enumerate(imgs):
        col = i % cols
        row = i // cols
        
        # Left edge alignment
        x = col * cell_w
        
        # Bottom line alignment
        y = row * cell_h + (max_h - img.height)
        
        grid.paste(img, (x, y))

    return grid

In [45]:
faces = [f for f in os.listdir("/Applications/HEXPLODATIONS/Utopia-Regular_8_dfont") if f.endswith(".dfont")]
for f in faces:
    print(f"\n=== {f} ===")
    face = freetype.Face(os.path.join("/Applications/HEXPLODATIONS/Utopia-Regular_8_dfont", f))
    face.select_size(0)
    try:
        grid = make_grid(face)
    except Exception as e:
        print(f"Error occurred while processing {f}: {e}")
    else:
        grid.save(f"../Utopia-Regular_8_images/{f}.png")


=== Utopia-Regular_8_2169_00000879.dfont ===

glyph 'A': 5×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'B': 4×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'C': 5×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'a': 4×4px, pitch=1, buffer=4 байт, pixel_mode=1

glyph 'b': 4×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'c': 4×4px, pitch=1, buffer=4 байт, pixel_mode=1

glyph '0': 4×5px, pitch=1, buffer=5 байт, pixel_mode=1

glyph '1': 2×5px, pitch=1, buffer=5 байт, pixel_mode=1

glyph '2': 4×5px, pitch=1, buffer=5 байт, pixel_mode=1

=== Utopia-Regular_8_1969_000007B1.dfont ===

glyph 'A': 5×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'B': 4×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'C': 5×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'a': 4×4px, pitch=1, buffer=4 байт, pixel_mode=1

glyph 'b': 4×6px, pitch=1, buffer=6 байт, pixel_mode=1

glyph 'c': 4×4px, pitch=1, buffer=4 байт, pixel_mode=1

glyph '0': 4×5px, pitch=1, buffer=5 байт, pixel_mode=1

glyph '1': 